# Adult Census Income: mixed-type Spark ML pipeline in OMLE

Demonstrate a raw DataFrame containing **integer, floating-point, and string columns**, including missing values. Fit a Spark ML preprocessing pipeline and classifier together, export the complete pipeline, inspect its interface and operators, validate, view, and verify predictions.

## Dataset and input contract

Load pinned [OpenML dataset 1590](https://www.openml.org/d/1590) through `sklearn.datasets.fetch_openml`. The first run needs internet; subsequent runs reuse `.cache/openml/`, excluded from Git. The returned metadata below records the dataset version, source checksum, attribution, and license. No dataset is committed or uploaded.

Use a deliberate subset of available predictors to keep the graph readable. Integer columns remain integer; continuous numeric columns are explicitly float64, including numeric measurements stored as integers by the source. Categorical columns are Python strings (`object` dtype), with `np.nan` for missing entries. These type and missing-marker normalizations define the input contract for both native and OMLE predictions; learned imputation, scaling, and encoding remain inside the exported graph.
This is an educational portability example, not a deployment recommendation.

**Setup:** install `requirements-spark.txt` in the notebook kernel environment. These examples use pandas 2.x because the released OMLE runtime mixed-column path requires writable arrays; pandas 3 can expose read-only arrays.

Use Java 17 and set `JAVA_HOME` before launching Jupyter. This recipe supports PySpark 3.5 and 4.x (validated with 3.5.9 and 4.3.0), running locally with two threads and uses the kernel interpreter for its workers. The final cell stops Spark. It uses the same OpenML data ID and split seed as the scikit-learn counterpart; independently fitted models need not agree across frameworks.

PySpark 4.3 requires Python 3.11 or newer.

In [ ]:
import os
import sys
import hashlib
import json
import platform
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import fetch_openml
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import SQLTransformer, Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.regression import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, log_loss, roc_auc_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


import omle
import omle_runtime as omr
from omle_convert import export_omle

SEED = 42
ATOL = 1e-4
RTOL = 1e-4
DATA_ID = 1590
integer_columns = ['age', 'education-num']
float_columns = ['capital-gain', 'capital-loss', 'hours-per-week']
categorical_columns = ['workclass', 'education', 'marital-status', 'occupation', 'native-country']
numeric_columns = integer_columns + float_columns
feature_columns = numeric_columns + categorical_columns
packages = ["numpy", "pandas", "scikit-learn", "omle", "omle-convert", "omle-runtime", "omle-viewer", "pyspark"]
environment = {"python": platform.python_version(), "packages": {p: version(p) for p in packages}}
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents]
                  if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Start Jupyter from omle-models or a notebook subdirectory.")
if int(version("pandas").split(".")[0]) >= 3:
    raise RuntimeError("Install requirements-spark.txt and restart the kernel; mixed-column inference requires pandas 2.x with the current OMLE runtime.")
display(environment)

## Load and inspect mixed types

Download by immutable dataset ID. Preserve the returned source metadata and compute a SHA-256 fingerprint of the normalized selected data and target. Display the input dtypes, example values, and missing-value counts before fitting.

In [ ]:
cache_dir = repo_root / ".cache" / "openml"
# Read source fields as strings; apply the explicit input contract below.
dataset = fetch_openml(data_id=DATA_ID, as_frame=True, parser="pandas", data_home=str(cache_dir),
                       read_csv_kwargs={"dtype": str})
source_metadata = {key: dataset.details.get(key) for key in
                   ["id", "name", "version", "licence", "creator", "citation", "url", "md5_checksum"]}
display(source_metadata)
X = dataset.data.loc[:, feature_columns].copy()
for column in integer_columns:
    values = pd.to_numeric(X[column], errors="raise")
    assert values.notna().all() and (values == np.floor(values)).all()
    X[column] = values.astype(np.int64)
for column in float_columns:
    X[column] = pd.to_numeric(X[column], errors="raise").astype(np.float64)
for column in categorical_columns:
    values = X[column].astype(object)
    values = values.map(lambda value: str(value).strip() if pd.notna(value) else np.nan)
    X[column] = values.replace("?", np.nan).astype(object)
target_values = dataset.target.astype(str).str.strip().str.rstrip(".")
assert set(target_values.unique()) == {"<=50K", ">50K"}
y = target_values.map({"<=50K": 0, ">50K": 1}).astype(np.int64)
class_names = ["income <=50K", "income >50K"]
assert not y.isna().any()
input_summary = pd.DataFrame({"dtype": X.dtypes.astype(str), "missing_count": X.isna().sum()})
display(input_summary)
display(X.head())
normalized_data = X.assign(__target__=y)
dataset_sha256 = hashlib.sha256(pd.util.hash_pandas_object(normalized_data, index=True).to_numpy().tobytes()).hexdigest()
print(f"Rows: {len(X)}; selected features: {len(X.columns)}; normalized data SHA-256: {dataset_sha256}")

## Split and fit Spark preprocessing

Use the same fixed 80/20 split as the scikit-learn example (stratified for classification). Train only on the training rows.

- Floating-point columns: Spark median `Imputer`; integer columns are complete and retain integer input types.
- String columns: `StringIndexer(handleInvalid="keep")`, an explicit SQL `COALESCE` fallback to the learned unknown index, then `OneHotEncoder(dropLast=True)`, fitted on the complete index domain including the reserved unknown index. Missing and unseen strings share that last index and therefore encode as an all-zero vector. Index-domain fitting uses only training vocabulary sizes; it adds no synthetic predictor-training rows or labels.
- `VectorAssembler` combines the branches. Linear models additionally use `StandardScaler`.

Map missing strings to Spark nulls; OMLE receives the corresponding raw DataFrame missing values. Stable row IDs preserve prediction alignment after distributed collection. The null/unknown policy is tested below rather than assumed.

The SQL fallback preserves Spark’s unknown-index behavior when exporting: OMLE string lookup emits NaN for an unmatched string, which `COALESCE` maps to the same index. The fallback indices are learned from training vocabularies and included in the exported graph.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (SparkSession.builder.master("local[2]")
         .appName("omle-models-mixed-adult")
         .config("spark.ui.enabled", "false")
         .config("spark.sql.shuffle.partitions", "2")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Explicit schemas retain long integers, doubles, and strings.
schema = StructType(
    [StructField("row_id", LongType(), False)]
    + [StructField(name, LongType(), False) for name in integer_columns]
    + [StructField(name, DoubleType(), True) for name in float_columns]
    + [StructField(name, StringType(), True) for name in categorical_columns]
    + [StructField("label", DoubleType(), False)]
)

def to_spark_frame(frame, labels=None):
    values = labels.to_numpy() if labels is not None else np.zeros(len(frame))
    rows = []
    for row_id, (_, row), label in zip(frame.index, frame.iterrows(), values):
        rows.append(tuple(
            [int(row_id)]
            + [int(row[name]) for name in integer_columns]
            + [float(row[name]) if pd.notna(row[name]) else None for name in float_columns]
            + [str(row[name]) if pd.notna(row[name]) else None for name in categorical_columns]
            + [float(label)]
        ))
    return spark.createDataFrame(rows, schema=schema)

train_df = to_spark_frame(X_train, y_train)
train_df.printSchema()
imputed_columns = [name + "_imputed" for name in float_columns]
indexed_columns = [name + "_indexed" for name in categorical_columns]
safe_indexed_columns = [name + "_safe_index" for name in categorical_columns]
encoded_columns = [name + "_encoded" for name in categorical_columns]
# One transformer per column keeps the exported outputs explicit.
stages = [Imputer(inputCols=[name], outputCols=[output], strategy="median")
          for name, output in zip(float_columns, imputed_columns)]
# Fit vocabulary only on training rows. Keep the unknown index explicit
# before encoding: OMLE LabelEncoder represents unmatched strings as NaN.
indexers = [StringIndexer(inputCol=name, outputCol=output,
                          handleInvalid="keep", stringOrderType="alphabetAsc").fit(train_df)
            for name, output in zip(categorical_columns, indexed_columns)]
unknown_indices = {name: len(indexer.labels)
                   for name, indexer in zip(categorical_columns, indexers)}
stages += indexers
fallbacks = [f"COALESCE(`{indexed}`, {unknown_indices[name]}.0) AS `{safe}`"
             for name, indexed, safe in zip(categorical_columns, indexed_columns, safe_indexed_columns)]
stages += [SQLTransformer(statement="SELECT *, " + ", ".join(fallbacks) + " FROM __THIS__")]
# Fit each encoder on the index domain, including its reserved unknown
# index. This uses only training vocabulary sizes, not synthetic labels.
# dropLast then consistently drops the unknown bucket for every column.
for name, safe, output in zip(categorical_columns, safe_indexed_columns, encoded_columns):
    index_domain = spark.createDataFrame(
        [(float(i),) for i in range(unknown_indices[name] + 1)], [safe]
    )
    stages.append(OneHotEncoder(inputCol=safe, outputCol=output, dropLast=True).fit(index_domain))
stages += [VectorAssembler(inputCols=integer_columns + imputed_columns + encoded_columns,
                           outputCol="assembled_features")]
stages += [RandomForestClassifier(featuresCol="assembled_features", labelCol="label",
                                  numTrees=30, maxDepth=5, seed=SEED)]
pipeline = SparkPipeline(stages=stages).fit(train_df)

# Collect predictions by ID, never relying on distributed row order.
def native_outputs(frame):
    rows = pipeline.transform(to_spark_frame(frame)).select("row_id", "prediction", "probability").collect()
    by_id = {row.row_id: row for row in rows}
    prediction = np.array([by_id[int(i)].prediction for i in frame.index], dtype=np.int64)
    probability = np.array([by_id[int(i)].probability.toArray() for i in frame.index])
    return prediction, probability
print(f"Training rows: {len(X_train)}; held-out rows: {len(X_test)}")

## Evaluate the native pipeline

Quality metrics describe the held-out prediction task. They do not establish conversion correctness; that is verified independently below.

In [ ]:
native_labels, native_probabilities = native_outputs(X_test)
classes = np.array([0, 1])
display(pd.DataFrame({"probability_column": [0, 1], "label": classes, "class_name": class_names}))
metrics = {
    "accuracy": float(accuracy_score(y_test, native_labels)),
    "log_loss": float(log_loss(y_test, native_probabilities, labels=classes)),
    "roc_auc_class_1": float(roc_auc_score(y_test, native_probabilities[:, 1])),
}
display(pd.Series(metrics, name="held_out_metrics"))
print(classification_report(y_test, native_labels, target_names=class_names, zero_division=0))

## Export the fitted Spark pipeline

Convert the entire fitted pipeline with its raw mixed-column training DataFrame. The exported graph includes imputation, string indexing, one-hot encoding, assembly, scaling when used, and the model. Embedded verification rows use training data; held-out and edge-case parity checks below are independent.

In [ ]:
output_dir = repo_root / "models" / "classification" / "adult"
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "spark-mixed-random-forest.omle"
export_omle(
    pipeline, str(model_path), dataset=train_df, feature_names=feature_columns,
    model_name="adult-spark-mixed-random-forest", model_version="0.1.0",
    random_state=SEED, n_verify=10, n_sample=3, verify_atol=ATOL, verify_rtol=RTOL,
)
print(f"Exported {model_path.relative_to(repo_root)} ({model_path.stat().st_size:,} bytes)")

## Inspect the exported model

Show the mixed-type input contract and output specifications. The graph accepts raw columns, not a pre-encoded numeric matrix. A dimension of `-1` represents a variable batch size.

In [ ]:
import omle

exported_model = omle.load(str(model_path))

print("Model inputs")
display(pd.DataFrame([
    {
        "name": spec.name,
        "dtype": spec.type.dtype.name if spec.type else None,
        "shape": spec.type.shape if spec.type else None,
    }
    for spec in exported_model.inputs
]))

print("Model outputs")
display(pd.DataFrame([
    {
        "name": spec.name,
        "dtype": spec.type.dtype.name if spec.type else None,
        "shape": spec.type.shape if spec.type else None,
        "role": spec.role.name,
    }
    for spec in exported_model.outputs
]))

### Operators included in the model

List actual graph operators, including nodes nested inside composite graphs when present.

In [ ]:
operator_versions = {item.namespace: item.version for item in exported_model.operator_imports}

def operator_rows(nodes, parent=""):
    for node in nodes:
        path = f"{parent}/{node.name}" if parent else node.name
        yield {"node": path, "namespace": node.domain, "operator": node.op,
               "version": operator_versions.get(node.domain)}
        if node.composite is not None:
            yield from operator_rows(node.composite.nodes, path)

operators = pd.DataFrame(operator_rows(exported_model.nodes))
display(operators)

### Validate the model

Structural validation checks the exported graph and operator declarations; errors stop execution. Numerical parity is checked below.

In [ ]:
validation = omle.validate(exported_model)
print(validation)
if not validation.is_valid:
    raise ValueError(f"Exported OMLE model is invalid:\n{validation}")

### Explore the interactive graph

Inspect the imputer, scaler, encoder, concatenation, and model nodes. Run the cell in JupyterLab or your IDE notebook editor; the viewer does not execute in GitHub’s static preview. No model upload is needed.

In [ ]:
from omle_viewer import show

viewer = show(exported_model, height=650)

## Verify raw DataFrame predictions

Send the same raw mixed-type DataFrame directly to both implementations, including its missing values. Require matching shapes, finite values, exact classification labels, and numeric agreement at the stated tolerances. Class probability order is checked against the native estimator above.

In [ ]:
runtime_model = omr.load(str(model_path))

def verify_predictions(frame):
    native, expected_proba = native_outputs(frame)
    actual = runtime_model.predict(frame)
    assert actual.shape == native.shape
    assert np.isfinite(actual).all()
    np.testing.assert_array_equal(actual, native)
    actual_proba = runtime_model.predict_proba(frame)
    assert actual_proba.shape == expected_proba.shape
    assert np.isfinite(actual_proba).all()
    assert ((actual_proba >= 0) & (actual_proba <= 1)).all()
    np.testing.assert_allclose(actual_proba, expected_proba, atol=ATOL, rtol=RTOL)
    np.testing.assert_allclose(actual_proba.sum(axis=1), 1.0, atol=ATOL, rtol=RTOL)
    return {"rows": len(frame), "label_agreement": float(np.mean(actual == native)),
            "max_absolute_probability_error": float(np.max(np.abs(actual_proba - expected_proba)))}
parity = verify_predictions(X_test)
parity.update({"atol": ATOL, "rtol": RTOL})
display(pd.Series(parity, name="held_out_conversion_fidelity"))
print("PASS: raw mixed-type test predictions match.")

## Check missing values and unseen categories

Construct separate probes from held-out rows without refitting. Inject a missing float, a missing string, and an unseen string category, then compare both inference paths. These are conversion checks, not additional quality measurements. Integer columns remain integers; missing numeric values are inserted into the floating-point branch.

In [ ]:
probes = pd.concat([X_test.iloc[[0]].copy() for _ in range(4)], ignore_index=True)
probe_names = ["baseline", "missing_numeric", "missing_categorical", "unseen_category"]
probes.loc[1, float_columns[0]] = np.nan
probes.loc[2, categorical_columns[0]] = np.nan
unknown_value = "__OMLE_UNSEEN_CATEGORY__"
assert unknown_value not in set(X_train[categorical_columns[0]].dropna())
probes.loc[3, categorical_columns[0]] = unknown_value
display(probes.assign(probe=probe_names))
edge_parity = verify_predictions(probes)
display(pd.Series(edge_parity, name="edge_case_conversion_fidelity"))
print("PASS: missing values and unseen categories match the native pipeline.")

## Save the run record

Record source identity/checksum, normalized-data fingerprint, column dtypes and selections, exact dependencies, split membership, fitted configuration, and both parity checks. Files remain local and ignored until explicitly selected for publication.

In [ ]:
report = {
    "environment": environment, "dataset": source_metadata,
    "normalized_dataset_sha256": dataset_sha256,
    "seed": SEED, "test_size": 0.2,
    "feature_names": list(X.columns), "input_dtypes": X.dtypes.astype(str).to_dict(),
    "integer_columns": integer_columns, "float_columns": float_columns,
    "categorical_columns": categorical_columns, "unknown_indices": unknown_indices,
    "categorical_missing_policy": "StringIndexer keep bucket with SQL coalesce to learned unknown index; full index-domain OHE dropLast maps unknown to all-zero",
    "train_indices": X_train.index.tolist(), "test_indices": X_test.index.tolist(),
    "model_parameters": {str(k): str(v) for k, v in [(f"{stage.__class__.__name__}.{param.name}", value)
                          for stage in pipeline.stages for param, value in stage.extractParamMap().items()]},
    "metrics": metrics, "parity": parity, "edge_parity": edge_parity,
}
report["class_names"] = class_names
report["classes"] = classes.tolist()
report_path = output_dir / "spark-mixed-random-forest-environment.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Saved {report_path.relative_to(repo_root)}")

## Stop Spark

Release the local JVM resources. If execution was interrupted earlier, run this cell manually.

In [ ]:
spark.stop()